In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler

# Configure OpenMP thread affinity before initializing PyTorch operations
# Limits execution to physical P-cores to eliminate E-core barrier stalls
NUM_PHYSICAL_CORES = 6
os.environ["OMP_NUM_THREADS"] = str(NUM_PHYSICAL_CORES)
os.environ["KMP_AFFINITY"] = "granularity=fine,compact,1,0"
os.environ["KMP_BLOCKTIME"] = "1"

torch.set_num_threads(NUM_PHYSICAL_CORES)

# Define Autoencoder Architecture
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# Optimized Dataset class using pre-allocated contiguous float32 memory
class OptimizedWeatherDataset(Dataset):
    def __init__(self, X_data):
        self.X = torch.from_numpy(X_data).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx]

# Feature definitions
features = [
    "temperature_C",
    "humidity_pct",
    "pressure_hPa",
    "dew_point_C",
    "pressure_trend",
    "solar_radiation_Wm2",
    "wind_speed_ms",
    "cloud_cover_pct",
    "wind_dir_sin",
    "wind_dir_cos",
    "cape",
    "et0_mm",
    "precip_mm"
]

# Hardware Target Selection: Select Intel Arc GPU ("xpu") if available
device = torch.device("xpu" if hasattr(torch, "xpu") and torch.xpu.is_available() else "cpu")
print("Device:", device)

if device.type == "xpu":
    print("XPU device:", torch.xpu.get_device_name(0))

model = Autoencoder(input_dim=len(features)).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Reduced chunksize prevents system RAM saturation and paging overhead
chunksize = 2_000_000
scaler = StandardScaler()
first_chunk = True
all_anomaly_flags = []

# Accelerated multi-threaded CSV reading via PyArrow engine
reader = pd.read_csv(
    r"C:\Users\abhin\Downloads\archive\Indian_Weather_Dataset.csv",
    chunksize=chunksize,
    usecols=features
)

for i, chunk in enumerate(reader):
    print(f"Processing chunk {i+1}...")
    # Extract feature matrix directly into contiguous float32 array
    X_mat = chunk[features].dropna().to_numpy(dtype=np.float32)

    if first_chunk:
        X_scaled = scaler.fit_transform(X_mat)
        first_chunk = False
    else:
        X_scaled = scaler.transform(X_mat)

    dataset = OptimizedWeatherDataset(X_scaled)

    # Multi-worker DataLoader with pinned memory for non-blocking transfers
    loader = DataLoader(
        dataset,
        batch_size=32768,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == "xpu"),
        drop_last=False
    )

    # Corrected Training Loop: Pass full mini-batch tensors
    model.train()
    for epoch in range(1):
        for xb in loader:
            xb = xb.to(device, non_blocking=True)  # Shape: [32768, 10]
            optimizer.zero_grad()
            recon = model(xb)
            loss = criterion(recon, xb)
            loss.backward()
            optimizer.step()

    # Optimized Evaluation Loop
    model.eval()
    errors = []
    score_loader = DataLoader(
        dataset,
        batch_size=65536,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "xpu")
    )

    with torch.no_grad():
        for xb in score_loader:
            xb = xb.to(device, non_blocking=True)
            recon = model(xb)
            batch_errors = torch.mean((recon - xb) ** 2, dim=1).cpu().numpy()
            errors.extend(batch_errors)

    threshold = np.quantile(errors, 0.99)
    flags = (errors > threshold).astype(int)
    all_anomaly_flags.extend(flags)

print("Total anomalies detected:", sum(all_anomaly_flags))

In [1]:
# ============================================================
# SKYGUARD V2 — CELL 1
# Environment & Intel Arc XPU Verification
# ============================================================

import sys
import os
import torch

print("=" * 60)
print("              SKYGUARD V2 - SYSTEM CHECK")
print("=" * 60)

# Python environment
print(f"Python executable : {sys.executable}")
print(f"Python version    : {sys.version.split()[0]}")

# PyTorch
print(f"PyTorch version   : {torch.__version__}")
print(f"Has XPU support  : {hasattr(torch, 'xpu')}")

# XPU verification
if not hasattr(torch, "xpu"):
    raise RuntimeError(
        "❌ This PyTorch installation does not contain XPU support."
    )

if not torch.xpu.is_available():
    raise RuntimeError(
        "❌ Intel XPU is NOT available.\n"
        "Check the Jupyter kernel and Intel GPU driver."
    )

# Device information
device = torch.device("xpu")

print(f"XPU available     : {torch.xpu.is_available()}")
print(f"XPU device count  : {torch.xpu.device_count()}")
print(f"XPU device        : {torch.xpu.get_device_name(0)}")

# Select device
torch.xpu.set_device(0)

print("-" * 60)
print("✅ SkyGuard V2 XPU is READY")
print(f"🚀 Using device   : {device}")
print("=" * 60)

              SKYGUARD V2 - SYSTEM CHECK
Python executable : C:\Users\abhin\miniconda3\python.exe
Python version    : 3.13.13
PyTorch version   : 2.13.0+xpu
Has XPU support  : True
XPU available     : True
XPU device count  : 1
XPU device        : Intel(R) Arc(TM) 130T GPU (8GB)
------------------------------------------------------------
✅ SkyGuard V2 XPU is READY
🚀 Using device   : xpu


In [2]:
# ============================================================
# SKYGUARD V2 — CELL 2
# Dataset Configuration & Validation
# ============================================================

import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Dataset path
# ------------------------------------------------------------

DATA_PATH = r"C:\Users\abhin\Downloads\archive\Indian_Weather_Dataset.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"\n❌ Dataset not found:\n{DATA_PATH}\n"
        "\nCheck the file path."
    )

print("=" * 60)
print("          SKYGUARD V2 - DATASET CHECK")
print("=" * 60)

print(f"Dataset path : {DATA_PATH}")

# ------------------------------------------------------------
# 2. Features used by the Autoencoder
# ------------------------------------------------------------

FEATURES = [
    "temperature_C",
    "humidity_pct",
    "pressure_hPa",
    "dew_point_C",
    "pressure_trend",
    "solar_radiation_Wm2",
    "wind_speed_ms",
    "cloud_cover_pct",
    "wind_dir_sin",
    "wind_dir_cos",
    "cape",
    "et0_mm",
    "precip_mm"
]

# ------------------------------------------------------------
# 3. Metadata columns
# ------------------------------------------------------------

METADATA_COLUMNS = [
    "datetime",
    "state",
    "city",
    "lat",
    "lon",
    "crops"
]

# ------------------------------------------------------------
# 4. Read only a small sample
# ------------------------------------------------------------

print("\nReading sample...")

sample = pd.read_csv(
    DATA_PATH,
    nrows=10_000
)

print(f"Sample rows : {len(sample):,}")
print(f"Columns     : {len(sample.columns)}")

# ------------------------------------------------------------
# 5. Check required columns
# ------------------------------------------------------------

required_columns = FEATURES + METADATA_COLUMNS

missing_columns = [
    col for col in required_columns
    if col not in sample.columns
]

if missing_columns:

    print("\n❌ Missing columns:")

    for col in missing_columns:
        print("   -", col)

    raise ValueError(
        "\nDataset does not contain all required columns."
    )

print("\n✅ All required columns are present.")

# ------------------------------------------------------------
# 6. Display feature information
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("MODEL FEATURES")
print("-" * 60)

for i, feature in enumerate(FEATURES, start=1):

    print(f"{i:2}. {feature}")

# ------------------------------------------------------------
# 7. Missing-value analysis
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("MISSING VALUES IN SAMPLE")
print("-" * 60)

missing = sample[FEATURES].isna().sum()

for feature, count in missing.items():

    percentage = (count / len(sample)) * 100

    print(
        f"{feature:25} "
        f"{count:8,} "
        f"({percentage:6.2f}%)"
    )

# ------------------------------------------------------------
# 8. Numeric conversion check
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("DATA TYPES")
print("-" * 60)

for feature in FEATURES:

    print(
        f"{feature:25} "
        f"{str(sample[feature].dtype)}"
    )

# ------------------------------------------------------------
# 9. Basic statistics
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("FEATURE STATISTICS")
print("-" * 60)

stats = sample[FEATURES].describe().T

display(
    stats[
        ["count", "mean", "std", "min", "max"]
    ].round(3)
)

# ------------------------------------------------------------
# 10. Dataset information
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET CHECK COMPLETE")
print("=" * 60)

print(f"Model features : {len(FEATURES)}")
print(f"Metadata       : {len(METADATA_COLUMNS)}")

print("\n✅ Cell 2 completed successfully.")

          SKYGUARD V2 - DATASET CHECK
Dataset path : C:\Users\abhin\Downloads\archive\Indian_Weather_Dataset.csv

Reading sample...
Sample rows : 10,000
Columns     : 23

✅ All required columns are present.

------------------------------------------------------------
MODEL FEATURES
------------------------------------------------------------
 1. temperature_C
 2. humidity_pct
 3. pressure_hPa
 4. dew_point_C
 5. pressure_trend
 6. solar_radiation_Wm2
 7. wind_speed_ms
 8. cloud_cover_pct
 9. wind_dir_sin
10. wind_dir_cos
11. cape
12. et0_mm
13. precip_mm

------------------------------------------------------------
MISSING VALUES IN SAMPLE
------------------------------------------------------------
temperature_C                    0 (  0.00%)
humidity_pct                     0 (  0.00%)
pressure_hPa                     0 (  0.00%)
dew_point_C                      0 (  0.00%)
pressure_trend                   0 (  0.00%)
solar_radiation_Wm2              0 (  0.00%)
wind_speed_ms       

,count,mean,std,min,max
temperature_C,10000.0,26.821,6.035,10.0,43.90
humidity_pct,10000.0,54.859,23.442,8.0,100.00
pressure_hPa,10000.0,978.757,4.974,962.6,990.40
dew_point_C,10000.0,15.224,5.840,-3.5,25.30
pressure_trend,10000.0,-0.000,0.639,-2.3,2.00
solar_radiation_Wm2,10000.0,222.057,299.841,0.0,1040.00
wind_speed_ms,10000.0,8.314,3.985,0.0,27.60
cloud_cover_pct,10000.0,40.355,40.315,0.0,100.00
wind_dir_sin,10000.0,-0.162,0.736,-1.0,1.00
wind_dir_cos,10000.0,0.094,0.650,-1.0,1.00



DATASET CHECK COMPLETE
Model features : 13
Metadata       : 6

✅ Cell 2 completed successfully.


In [3]:
# ============================================================
# SKYGUARD V2 — CELL 3
# High-Speed Chunked Preprocessing
# ============================================================

import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("=" * 60)
print("       SKYGUARD V2 - PREPROCESSING")
print("=" * 60)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CHUNK_SIZE = 2_000_000

# We need these columns for ML + final output
DATA_COLUMNS = FEATURES + METADATA_COLUMNS

print(f"Chunk size       : {CHUNK_SIZE:,}")
print(f"Columns loaded   : {len(DATA_COLUMNS)}")

# ------------------------------------------------------------
# First pass
# ------------------------------------------------------------
# We don't load the entire dataset.
# We inspect it chunk-by-chunk to determine:
#   - valid rows
#   - constant features
#   - feature statistics
# ------------------------------------------------------------

feature_min = np.full(len(FEATURES), np.inf, dtype=np.float64)
feature_max = np.full(len(FEATURES), -np.inf, dtype=np.float64)

total_rows = 0
valid_rows = 0
invalid_rows = 0

reader = pd.read_csv(
    DATA_PATH,
    usecols=DATA_COLUMNS,
    chunksize=CHUNK_SIZE
)

for chunk_number, chunk in enumerate(reader, start=1):

    print(
        f"\rScanning chunk {chunk_number}...",
        end="",
        flush=True
    )

    # --------------------------------------------------------
    # Convert ML features to float32
    # --------------------------------------------------------

    X = chunk[FEATURES].to_numpy(
        dtype=np.float32,
        copy=True
    )

    total_rows += len(X)

    # --------------------------------------------------------
    # Detect NaN / infinity
    # --------------------------------------------------------

    valid_mask = np.isfinite(X).all(axis=1)

    valid_count = valid_mask.sum()

    valid_rows += valid_count
    invalid_rows += len(X) - valid_count

    if valid_count == 0:
        continue

    X_valid = X[valid_mask]

    # --------------------------------------------------------
    # Update global min/max
    # --------------------------------------------------------

    feature_min = np.minimum(
        feature_min,
        np.min(X_valid, axis=0)
    )

    feature_max = np.maximum(
        feature_max,
        np.max(X_valid, axis=0)
    )

    del X, X_valid, chunk
    gc.collect()

print("\n")

# ------------------------------------------------------------
# Detect constant features
# ------------------------------------------------------------

constant_features = []

for i, feature in enumerate(FEATURES):

    if np.isclose(
        feature_min[i],
        feature_max[i]
    ):
        constant_features.append(feature)

# ------------------------------------------------------------
# Remove constant features
# ------------------------------------------------------------

ACTIVE_FEATURES = [
    feature
    for feature in FEATURES
    if feature not in constant_features
]

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("-" * 60)
print("DATASET SUMMARY")
print("-" * 60)

print(f"Total rows scanned       : {total_rows:,}")
print(f"Valid rows               : {valid_rows:,}")
print(f"Invalid rows             : {invalid_rows:,}")
print(
    f"Valid percentage         : "
    f"{valid_rows / total_rows * 100:.2f}%"
)

print("\n" + "-" * 60)
print("CONSTANT FEATURES")
print("-" * 60)

if constant_features:

    for feature in constant_features:
        idx = FEATURES.index(feature)

        print(
            f"{feature:25} "
            f"value = {feature_min[idx]:.4f}"
        )

else:

    print("None detected.")

print("\n" + "-" * 60)
print("ACTIVE MODEL FEATURES")
print("-" * 60)

for i, feature in enumerate(
    ACTIVE_FEATURES,
    start=1
):

    print(f"{i:2}. {feature}")

# ------------------------------------------------------------
# Store feature count
# ------------------------------------------------------------

INPUT_DIM = len(ACTIVE_FEATURES)

print("\n" + "=" * 60)

print(
    f"Original features : {len(FEATURES)}"
)

print(
    f"Active features   : {INPUT_DIM}"
)

print("=" * 60)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

if valid_rows == 0:

    raise RuntimeError(
        "❌ No valid rows were found."
    )

if INPUT_DIM < 2:

    raise RuntimeError(
        "❌ Too few active features remain."
    )

print("\n✅ Cell 3 completed successfully.")

       SKYGUARD V2 - PREPROCESSING
Chunk size       : 2,000,000
Columns loaded   : 19
Scanning chunk 24...

------------------------------------------------------------
DATASET SUMMARY
------------------------------------------------------------
Total rows scanned       : 46,082,160
Valid rows               : 46,082,160
Invalid rows             : 0
Valid percentage         : 100.00%

------------------------------------------------------------
CONSTANT FEATURES
------------------------------------------------------------
cape                      value = 0.0000

------------------------------------------------------------
ACTIVE MODEL FEATURES
------------------------------------------------------------
 1. temperature_C
 2. humidity_pct
 3. pressure_hPa
 4. dew_point_C
 5. pressure_trend
 6. solar_radiation_Wm2
 7. wind_speed_ms
 8. cloud_cover_pct
 9. wind_dir_sin
10. wind_dir_cos
11. et0_mm
12. precip_mm

Original features : 13
Active features   : 12

✅ Cell 3 completed successfully

In [4]:
# ============================================================
# SKYGUARD V2 — CELL 4
# XPU-Optimized Autoencoder
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

print("=" * 60)
print("       SKYGUARD V2 - MODEL INITIALIZATION")
print("=" * 60)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

if not torch.xpu.is_available():
    raise RuntimeError(
        "❌ XPU is not available. "
        "Do not continue until Intel Arc XPU is detected."
    )

device = torch.device("xpu")

print(f"Device       : {device}")
print(f"GPU          : {torch.xpu.get_device_name(0)}")
print(f"Features     : {INPUT_DIM}")

# ------------------------------------------------------------
# Autoencoder
# ------------------------------------------------------------

class SkyGuardAutoencoder(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(

            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 16)
        )

        # Decoder
        self.decoder = nn.Sequential(

            nn.Linear(16, 32),
            nn.ReLU(),

            nn.Linear(32, 64),
            nn.ReLU(),

            nn.Linear(64, input_dim)
        )

    def forward(self, x):

        encoded = self.encoder(x)

        reconstructed = self.decoder(encoded)

        return reconstructed


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

model = SkyGuardAutoencoder(
    input_dim=INPUT_DIM
).to(device)


# ------------------------------------------------------------
# Loss
# ------------------------------------------------------------

criterion = nn.MSELoss()


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)


# ------------------------------------------------------------
# Mixed precision
# ------------------------------------------------------------

USE_BF16 = False

try:

    if torch.xpu.is_bf16_supported():

        USE_BF16 = True

except Exception:

    USE_BF16 = False


print(f"BF16 support : {USE_BF16}")


# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("-" * 60)

print(
    f"Total parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_parameters:,}"
)

print(
    f"Input dimensions     : "
    f"{INPUT_DIM}"
)

print(
    f"Latent dimensions    : "
    f"16"
)

print("-" * 60)

print("Active features:")

for feature in ACTIVE_FEATURES:

    print(f"  • {feature}")

print("=" * 60)

print("✅ SkyGuard V2 model initialized successfully.")
print("🚀 Ready for XPU training.")

       SKYGUARD V2 - MODEL INITIALIZATION
Device       : xpu
GPU          : Intel(R) Arc(TM) 130T GPU (8GB)
Features     : 12
BF16 support : True
------------------------------------------------------------
Total parameters     : 6,876
Trainable parameters : 6,876
Input dimensions     : 12
Latent dimensions    : 16
------------------------------------------------------------
Active features:
  • temperature_C
  • humidity_pct
  • pressure_hPa
  • dew_point_C
  • pressure_trend
  • solar_radiation_Wm2
  • wind_speed_ms
  • cloud_cover_pct
  • wind_dir_sin
  • wind_dir_cos
  • et0_mm
  • precip_mm
✅ SkyGuard V2 model initialized successfully.
🚀 Ready for XPU training.


In [5]:
# ============================================================
# SKYGUARD V2 — CELL 5
# XPU Training Data Pipeline
# ============================================================

import gc
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

print("=" * 60)
print("        SKYGUARD V2 - XPU TRAINING PIPELINE")
print("=" * 60)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TRAIN_CHUNK_SIZE = 2_000_000

# Number of samples used for training.
# We don't need all 46 million rows to learn normal patterns.
MAX_TRAIN_SAMPLES = 2_000_000

BATCH_SIZE = 65_536

EPOCHS = 5

LEARNING_RATE = 1e-3

RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Training samples target : {MAX_TRAIN_SAMPLES:,}")
print(f"Batch size              : {BATCH_SIZE:,}")
print(f"Epochs                  : {EPOCHS}")
print(f"Learning rate            : {LEARNING_RATE}")

# ------------------------------------------------------------
# Read representative training data
# ------------------------------------------------------------

print("\nReading training data...")

training_parts = []

samples_collected = 0

reader = pd.read_csv(
    DATA_PATH,
    usecols=ACTIVE_FEATURES,
    chunksize=TRAIN_CHUNK_SIZE
)

for chunk_number, chunk in enumerate(reader, start=1):

    print(
        f"Reading chunk {chunk_number}...",
        end="\r",
        flush=True
    )

    X = chunk.to_numpy(
        dtype=np.float32,
        copy=True
    )

    # --------------------------------------------------------
    # Remove invalid rows
    # --------------------------------------------------------

    valid_mask = np.isfinite(X).all(axis=1)

    X = X[valid_mask]

    if len(X) == 0:
        continue

    # --------------------------------------------------------
    # Randomly sample from chunk
    # --------------------------------------------------------

    remaining = MAX_TRAIN_SAMPLES - samples_collected

    if len(X) > remaining:

        indices = np.random.choice(
            len(X),
            size=remaining,
            replace=False
        )

        X = X[indices]

    training_parts.append(X)

    samples_collected += len(X)

    if samples_collected >= MAX_TRAIN_SAMPLES:
        break

print("\n")

# ------------------------------------------------------------
# Combine training samples
# ------------------------------------------------------------

X_train = np.concatenate(
    training_parts,
    axis=0
)

del training_parts

gc.collect()

# ------------------------------------------------------------
# Shuffle
# ------------------------------------------------------------

np.random.shuffle(X_train)

print(
    f"Training samples loaded : "
    f"{len(X_train):,}"
)

# ------------------------------------------------------------
# Fit scaler on training data
# ------------------------------------------------------------

print("\nFitting StandardScaler...")

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
).astype(np.float32)

print("✅ Scaling complete.")

# ------------------------------------------------------------
# Convert to PyTorch tensor
# ------------------------------------------------------------

X_tensor = torch.from_numpy(
    X_train
)

# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

train_dataset = TensorDataset(
    X_tensor
)

# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

# ------------------------------------------------------------
# Free unnecessary NumPy memory
# ------------------------------------------------------------

del X_train
del X_tensor

gc.collect()

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\n" + "=" * 60)

print("TRAINING PIPELINE READY")

print(f"Samples : {len(train_dataset):,}")
print(f"Batches : {len(train_loader):,}")
print(f"Batch   : {BATCH_SIZE:,}")

print(f"Device  : {device}")

print(f"BF16    : {USE_BF16}")

print("=" * 60)

print("✅ Cell 5 completed successfully.")

        SKYGUARD V2 - XPU TRAINING PIPELINE
Training samples target : 2,000,000
Batch size              : 65,536
Epochs                  : 5
Learning rate            : 0.001

Reading training data...
Reading chunk 1...

Training samples loaded : 2,000,000

Fitting StandardScaler...
✅ Scaling complete.

TRAINING PIPELINE READY
Samples : 2,000,000
Batches : 30
Batch   : 65,536
Device  : xpu
BF16    : True
✅ Cell 5 completed successfully.


In [6]:
# ============================================================
# SKYGUARD V2 — CELL 6
# BF16 XPU Autoencoder Training
# ============================================================

import time
import torch

print("=" * 60)
print("          SKYGUARD V2 - MODEL TRAINING")
print("=" * 60)

# ------------------------------------------------------------
# Make sure model is on XPU
# ------------------------------------------------------------

model = model.to(device)

print(f"Device : {device}")
print(f"BF16   : {USE_BF16}")
print(f"Epochs : {EPOCHS}")
print()

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

training_history = []

for epoch in range(EPOCHS):

    model.train()

    epoch_start = time.perf_counter()

    total_loss = 0.0
    batch_count = 0

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print("-" * 60)

    for batch_idx, batch in enumerate(train_loader, start=1):

        # ----------------------------------------------------
        # Get batch
        # ----------------------------------------------------

        xb = batch[0].to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Clear gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        if USE_BF16:

            with torch.autocast(
                device_type="xpu",
                dtype=torch.bfloat16
            ):

                reconstruction = model(xb)

                loss = criterion(
                    reconstruction,
                    xb
                )

        else:

            reconstruction = model(xb)

            loss = criterion(
                reconstruction,
                xb
            )

        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        loss.backward()

        optimizer.step()

        # ----------------------------------------------------
        # Track loss
        # ----------------------------------------------------

        loss_value = loss.detach().float().item()

        total_loss += loss_value
        batch_count += 1

        # ----------------------------------------------------
        # Progress display
        # ----------------------------------------------------

        if (
            batch_idx == 1
            or batch_idx % 5 == 0
            or batch_idx == len(train_loader)
        ):

            average_loss = (
                total_loss / batch_count
            )

            elapsed = (
                time.perf_counter()
                - epoch_start
            )

            print(
                f"\rBatch "
                f"{batch_idx:02d}/{len(train_loader)} | "
                f"Loss: {loss_value:.6f} | "
                f"Avg: {average_loss:.6f} | "
                f"Time: {elapsed:.1f}s",
                end="",
                flush=True
            )

    # --------------------------------------------------------
    # Synchronize XPU
    # --------------------------------------------------------

    if device.type == "xpu":
        torch.xpu.synchronize()

    epoch_time = (
        time.perf_counter()
        - epoch_start
    )

    epoch_loss = (
        total_loss / batch_count
    )

    training_history.append(
        epoch_loss
    )

    print()

    print(
        f"Epoch {epoch + 1} completed | "
        f"Loss: {epoch_loss:.6f} | "
        f"Time: {epoch_time:.2f}s"
    )

# ------------------------------------------------------------
# Training complete
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print("\nTraining loss:")

for i, loss_value in enumerate(
    training_history,
    start=1
):

    print(
        f"Epoch {i}: "
        f"{loss_value:.6f}"
    )

print("\n✅ Autoencoder successfully trained on Intel Arc XPU.")

          SKYGUARD V2 - MODEL TRAINING
Device : xpu
BF16   : True
Epochs : 5


Epoch 1/5
------------------------------------------------------------
Batch 30/30 | Loss: 0.839230 | Avg: 0.953316 | Time: 23.7s
Epoch 1 completed | Loss: 0.953316 | Time: 23.85s

Epoch 2/5
------------------------------------------------------------
Batch 30/30 | Loss: 0.458281 | Avg: 0.610156 | Time: 19.4s
Epoch 2 completed | Loss: 0.610156 | Time: 19.47s

Epoch 3/5
------------------------------------------------------------
Batch 30/30 | Loss: 0.269902 | Avg: 0.336979 | Time: 19.5s
Epoch 3 completed | Loss: 0.336979 | Time: 19.68s

Epoch 4/5
------------------------------------------------------------
Batch 30/30 | Loss: 0.227027 | Avg: 0.245454 | Time: 19.4s
Epoch 4 completed | Loss: 0.245454 | Time: 19.56s

Epoch 5/5
------------------------------------------------------------
Batch 30/30 | Loss: 0.165861 | Avg: 0.200641 | Time: 19.2s
Epoch 5 completed | Loss: 0.200641 | Time: 19.34s

TRAINING COMPLET

In [7]:
# ============================================================
# SKYGUARD V2 — CELL 7
# Validation & Global Anomaly Threshold
# ============================================================

import gc
import time
import numpy as np
import pandas as pd
import torch

print("=" * 60)
print("       SKYGUARD V2 - VALIDATION")
print("=" * 60)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

VALIDATION_SAMPLES = 500_000
VALIDATION_CHUNK_SIZE = 2_000_000

print(
    f"Validation samples : "
    f"{VALIDATION_SAMPLES:,}"
)

print(
    f"Device             : "
    f"{device}"
)

# ------------------------------------------------------------
# Collect validation data
# ------------------------------------------------------------

validation_parts = []

samples_collected = 0

reader = pd.read_csv(
    DATA_PATH,
    usecols=ACTIVE_FEATURES,
    chunksize=VALIDATION_CHUNK_SIZE
)

for chunk_number, chunk in enumerate(
    reader,
    start=1
):

    print(
        f"Reading validation chunk {chunk_number}...",
        end="\r",
        flush=True
    )

    X = chunk.to_numpy(
        dtype=np.float32,
        copy=True
    )

    valid_mask = np.isfinite(X).all(axis=1)

    X = X[valid_mask]

    if len(X) == 0:
        continue

    remaining = (
        VALIDATION_SAMPLES
        - samples_collected
    )

    if len(X) > remaining:

        # Use a different random seed from training
        rng = np.random.default_rng(
            12345
        )

        indices = rng.choice(
            len(X),
            size=remaining,
            replace=False
        )

        X = X[indices]

    validation_parts.append(X)

    samples_collected += len(X)

    if samples_collected >= VALIDATION_SAMPLES:
        break

print("\n")

# ------------------------------------------------------------
# Combine validation data
# ------------------------------------------------------------

X_validation = np.concatenate(
    validation_parts,
    axis=0
)

del validation_parts

gc.collect()

print(
    f"Validation rows loaded : "
    f"{len(X_validation):,}"
)

# ------------------------------------------------------------
# IMPORTANT:
# Use the scaler fitted during Cell 5
# ------------------------------------------------------------

X_validation = scaler.transform(
    X_validation
).astype(np.float32)

# ------------------------------------------------------------
# Convert to tensor
# ------------------------------------------------------------

validation_tensor = torch.from_numpy(
    X_validation
)

del X_validation

gc.collect()

# ------------------------------------------------------------
# Calculate reconstruction errors
# ------------------------------------------------------------

validation_errors = []

validation_batch_size = 65_536

model.eval()

start_time = time.perf_counter()

print("\nCalculating reconstruction errors...")

with torch.no_grad():

    for start in range(
        0,
        len(validation_tensor),
        validation_batch_size
    ):

        end = min(
            start + validation_batch_size,
            len(validation_tensor)
        )

        xb = validation_tensor[
            start:end
        ].to(
            device,
            non_blocking=True
        )

        # BF16 inference
        if USE_BF16:

            with torch.autocast(
                device_type="xpu",
                dtype=torch.bfloat16
            ):

                reconstruction = model(xb)

        else:

            reconstruction = model(xb)

        errors = torch.mean(
            (reconstruction - xb) ** 2,
            dim=1
        )

        validation_errors.append(
            errors.float().cpu().numpy()
        )

        print(
            f"\rProcessed "
            f"{end:,}/{len(validation_tensor):,}",
            end="",
            flush=True
        )

# ------------------------------------------------------------
# Combine errors
# ------------------------------------------------------------

validation_errors = np.concatenate(
    validation_errors
)

if device.type == "xpu":
    torch.xpu.synchronize()

elapsed = (
    time.perf_counter()
    - start_time
)

print("\n")

print(
    f"Validation inference time : "
    f"{elapsed:.2f} seconds"
)

# ------------------------------------------------------------
# Error statistics
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("RECONSTRUCTION ERROR DISTRIBUTION")
print("-" * 60)

percentiles = [
    50,
    75,
    90,
    95,
    97,
    98,
    99,
    99.5,
    99.9
]

for p in percentiles:

    value = np.percentile(
        validation_errors,
        p
    )

    print(
        f"{p:5.1f}% percentile : "
        f"{value:.6f}"
    )

# ------------------------------------------------------------
# Global threshold
# ------------------------------------------------------------

ANOMALY_PERCENTILE = 99.0

ANOMALY_THRESHOLD = np.percentile(
    validation_errors,
    ANOMALY_PERCENTILE
)

print("\n" + "=" * 60)

print(
    f"Global anomaly threshold : "
    f"{ANOMALY_THRESHOLD:.6f}"
)

print(
    f"Threshold percentile     : "
    f"{ANOMALY_PERCENTILE}%"
)

# ------------------------------------------------------------
# Validation anomaly count
# ------------------------------------------------------------

validation_flags = (
    validation_errors
    > ANOMALY_THRESHOLD
)

validation_anomaly_count = (
    validation_flags.sum()
)

validation_anomaly_percentage = (
    validation_anomaly_count
    / len(validation_errors)
    * 100
)

print(
    f"Validation anomalies     : "
    f"{validation_anomaly_count:,}"
)

print(
    f"Validation anomaly rate  : "
    f"{validation_anomaly_percentage:.2f}%"
)

print("=" * 60)

# ------------------------------------------------------------
# Save validation errors for later analysis
# ------------------------------------------------------------

np.save(
    "skyguard_validation_errors.npy",
    validation_errors
)

print(
    "\n✅ Cell 7 completed successfully."
)

print(
    "Saved: skyguard_validation_errors.npy"
)

       SKYGUARD V2 - VALIDATION
Validation samples : 500,000
Device             : xpu
Reading validation chunk 1...

Validation rows loaded : 500,000

Calculating reconstruction errors...
Processed 500,000/500,000

Validation inference time : 0.94 seconds

------------------------------------------------------------
RECONSTRUCTION ERROR DISTRIBUTION
------------------------------------------------------------
 50.0% percentile : 0.130197
 75.0% percentile : 0.212331
 90.0% percentile : 0.312192
 95.0% percentile : 0.385389
 97.0% percentile : 0.443187
 98.0% percentile : 0.492170
 99.0% percentile : 0.590856
 99.5% percentile : 0.717426
 99.9% percentile : 1.245754

Global anomaly threshold : 0.590856
Threshold percentile     : 99.0%
Validation anomalies     : 5,000
Validation anomaly rate  : 1.00%

✅ Cell 7 completed successfully.
Saved: skyguard_validation_errors.npy


In [8]:
# ============================================================
# SKYGUARD V2 — CELL 8
# Sensor-Level Anomaly Explanation
# ============================================================

import numpy as np
import torch

print("=" * 60)
print("       SKYGUARD V2 - SENSOR EXPLANATION")
print("=" * 60)

# ------------------------------------------------------------
# Feature importance based on reconstruction error
# ------------------------------------------------------------

model.eval()

# Use the validation data again
# validation_tensor was created in Cell 7

feature_errors = []

batch_size = 65_536

with torch.no_grad():

    for start in range(
        0,
        len(validation_tensor),
        batch_size
    ):

        end = min(
            start + batch_size,
            len(validation_tensor)
        )

        xb = validation_tensor[
            start:end
        ].to(
            device,
            non_blocking=True
        )

        if USE_BF16:

            with torch.autocast(
                device_type="xpu",
                dtype=torch.bfloat16
            ):

                reconstruction = model(xb)

        else:

            reconstruction = model(xb)

        # ----------------------------------------------------
        # Per-feature squared reconstruction error
        # ----------------------------------------------------

        errors = (
            reconstruction - xb
        ) ** 2

        feature_errors.append(
            errors.float()
            .cpu()
            .numpy()
        )

# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

feature_errors = np.concatenate(
    feature_errors,
    axis=0
)

# ------------------------------------------------------------
# Average error for each sensor
# ------------------------------------------------------------

mean_feature_errors = np.mean(
    feature_errors,
    axis=0
)

# ------------------------------------------------------------
# Create feature ranking
# ------------------------------------------------------------

feature_ranking = sorted(
    zip(
        ACTIVE_FEATURES,
        mean_feature_errors
    ),
    key=lambda x: x[1],
    reverse=True
)

print("\n" + "-" * 60)
print("AVERAGE SENSOR RECONSTRUCTION ERROR")
print("-" * 60)

for rank, (feature, error) in enumerate(
    feature_ranking,
    start=1
):

    print(
        f"{rank:2}. "
        f"{feature:25} "
        f"{error:.6f}"
    )

# ------------------------------------------------------------
# Calculate contribution percentage
# ------------------------------------------------------------

total_error = mean_feature_errors.sum()

feature_contributions = (
    mean_feature_errors
    / total_error
    * 100
)

# ------------------------------------------------------------
# Contribution table
# ------------------------------------------------------------

sensor_contributions = pd.DataFrame({
    "feature": ACTIVE_FEATURES,
    "mean_error": mean_feature_errors,
    "contribution_percent": feature_contributions
})

sensor_contributions = (
    sensor_contributions
    .sort_values(
        "mean_error",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "-" * 60)
print("SENSOR CONTRIBUTION")
print("-" * 60)

display(
    sensor_contributions.round(4)
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

sensor_contributions.to_csv(
    "skyguard_sensor_contributions.csv",
    index=False
)

# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del feature_errors

print("\n" + "=" * 60)
print("✅ SENSOR EXPLANATION COMPLETE")
print("=" * 60)

print(
    "\nSaved: skyguard_sensor_contributions.csv"
)

       SKYGUARD V2 - SENSOR EXPLANATION

------------------------------------------------------------
AVERAGE SENSOR RECONSTRUCTION ERROR
------------------------------------------------------------
 1. pressure_trend            0.349456
 2. wind_dir_cos              0.269780
 3. cloud_cover_pct           0.242239
 4. wind_speed_ms             0.205902
 5. wind_dir_sin              0.185353
 6. temperature_C             0.170299
 7. pressure_hPa              0.133414
 8. precip_mm                 0.095892
 9. humidity_pct              0.089668
10. solar_radiation_Wm2       0.087356
11. dew_point_C               0.081287
12. et0_mm                    0.030511

------------------------------------------------------------
SENSOR CONTRIBUTION
------------------------------------------------------------


,feature,mean_error,contribution_percent
0,pressure_trend,0.3495,18.002501
1,wind_dir_cos,0.2698,13.897900
2,cloud_cover_pct,0.2422,12.479100
3,wind_speed_ms,0.2059,10.607200
4,wind_dir_sin,0.1854,9.548600
5,temperature_C,0.1703,8.773100
6,pressure_hPa,0.1334,6.872900
7,precip_mm,0.0959,4.939900
8,humidity_pct,0.0897,4.619300
9,solar_radiation_Wm2,0.0874,4.500200



✅ SENSOR EXPLANATION COMPLETE

Saved: skyguard_sensor_contributions.csv


In [9]:
# ============================================================
# SKYGUARD V2 — CELL 9
# FULL DATASET ANOMALY DETECTION
# ============================================================

import os
import gc
import time
import numpy as np
import pandas as pd
import torch

print("=" * 60)
print("       SKYGUARD V2 - FULL DATASET INFERENCE")
print("=" * 60)

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CHUNK_SIZE = 2_000_000
BATCH_SIZE = 65_536

OUTPUT_FILE = "skyguard_v2_results.csv"

print(f"Dataset              : {DATA_PATH}")
print(f"Output               : {OUTPUT_FILE}")
print(f"Chunk size           : {CHUNK_SIZE:,}")
print(f"Batch size           : {BATCH_SIZE:,}")
print(f"Device               : {device}")
print(f"Anomaly threshold    : {ANOMALY_THRESHOLD:.6f}")

# ------------------------------------------------------------
# Columns
# ------------------------------------------------------------

METADATA_COLUMNS = [
    "datetime",
    "state",
    "city",
    "lat",
    "lon",
    "crops"
]

OUTPUT_COLUMNS = (
    METADATA_COLUMNS
    + ACTIVE_FEATURES
    + [
        "anomaly_score",
        "is_anomaly",
        "severity",
        "dominant_sensor"
    ]
)

# ------------------------------------------------------------
# Remove previous output
# ------------------------------------------------------------

if os.path.exists(OUTPUT_FILE):

    os.remove(OUTPUT_FILE)

    print(
        "\nPrevious output file removed."
    )

# ------------------------------------------------------------
# Prepare CSV reader
# ------------------------------------------------------------

READ_COLUMNS = (
    METADATA_COLUMNS
    + ACTIVE_FEATURES
)

reader = pd.read_csv(
    DATA_PATH,
    usecols=READ_COLUMNS,
    chunksize=CHUNK_SIZE
)

# ------------------------------------------------------------
# Counters
# ------------------------------------------------------------

total_rows = 0
total_anomalies = 0
chunk_number = 0

start_total = time.perf_counter()

first_write = True

# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model.eval()

# ------------------------------------------------------------
# Process chunks
# ------------------------------------------------------------

for chunk in reader:

    chunk_number += 1

    chunk_start = time.perf_counter()

    print("\n" + "=" * 60)

    print(
        f"Processing chunk "
        f"{chunk_number}"
    )

    # --------------------------------------------------------
    # Extract features
    # --------------------------------------------------------

    X = chunk[
        ACTIVE_FEATURES
    ].to_numpy(
        dtype=np.float32
    )

    # --------------------------------------------------------
    # Handle invalid rows
    # --------------------------------------------------------

    valid_mask = np.isfinite(X).all(
        axis=1
    )

    # We keep only valid rows
    # for model inference.

    valid_indices = np.where(
        valid_mask
    )[0]

    X_valid = X[
        valid_mask
    ]

    # --------------------------------------------------------
    # Scale using training scaler
    # --------------------------------------------------------

    X_scaled = scaler.transform(
        X_valid
    ).astype(
        np.float32,
        copy=False
    )

    # --------------------------------------------------------
    # Tensor
    # --------------------------------------------------------

    X_tensor = torch.from_numpy(
        X_scaled
    )

    # --------------------------------------------------------
    # Result containers
    # --------------------------------------------------------

    chunk_scores = np.empty(
        len(X_valid),
        dtype=np.float32
    )

    chunk_dominant_sensor = np.empty(
        len(X_valid),
        dtype=object
    )

    # --------------------------------------------------------
    # Inference
    # --------------------------------------------------------

    with torch.no_grad():

        for start in range(
            0,
            len(X_tensor),
            BATCH_SIZE
        ):

            end = min(
                start + BATCH_SIZE,
                len(X_tensor)
            )

            xb = X_tensor[
                start:end
            ].to(
                device,
                non_blocking=True
            )

            # ------------------------------------------------
            # XPU BF16 inference
            # ------------------------------------------------

            if USE_BF16:

                with torch.autocast(
                    device_type="xpu",
                    dtype=torch.bfloat16
                ):

                    reconstruction = model(
                        xb
                    )

            else:

                reconstruction = model(
                    xb
                )

            # ------------------------------------------------
            # Per-sensor reconstruction error
            # ------------------------------------------------

            sensor_error = (
                reconstruction - xb
            ) ** 2

            # ------------------------------------------------
            # Overall anomaly score
            # ------------------------------------------------

            scores = torch.mean(
                sensor_error,
                dim=1
            )

            chunk_scores[
                start:end
            ] = (
                scores
                .float()
                .cpu()
                .numpy()
            )

            # ------------------------------------------------
            # Dominant sensor
            # ------------------------------------------------

            dominant_indices = torch.argmax(
                sensor_error,
                dim=1
            )

            dominant_indices = (
                dominant_indices
                .cpu()
                .numpy()
            )

            for local_i, feature_i in enumerate(
                dominant_indices
            ):

                chunk_dominant_sensor[
                    start + local_i
                ] = ACTIVE_FEATURES[
                    feature_i
                ]

    # --------------------------------------------------------
    # Create result dataframe
    # --------------------------------------------------------

    valid_chunk = chunk.iloc[
        valid_indices
    ].copy()

    valid_chunk[
        "anomaly_score"
    ] = chunk_scores

    valid_chunk[
        "is_anomaly"
    ] = (
        chunk_scores
        > ANOMALY_THRESHOLD
    ).astype(
        np.int8
    )

    valid_chunk[
        "dominant_sensor"
    ] = chunk_dominant_sensor

    # --------------------------------------------------------
    # Severity
    # --------------------------------------------------------

    scores = chunk_scores

    severity = np.full(
        len(scores),
        "Normal",
        dtype=object
    )

    severity[
        scores > ANOMALY_THRESHOLD
    ] = "Warning"

    severity[
        scores > ANOMALY_THRESHOLD * 1.5
    ] = "High"

    severity[
        scores > ANOMALY_THRESHOLD * 2.5
    ] = "Critical"

    valid_chunk[
        "severity"
    ] = severity

    # --------------------------------------------------------
    # Reorder columns
    # --------------------------------------------------------

    valid_chunk = valid_chunk[
        OUTPUT_COLUMNS
    ]

    # --------------------------------------------------------
    # Write incrementally
    # --------------------------------------------------------

    valid_chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_write else "a",
        header=first_write,
        index=False
    )

    first_write = False

    # --------------------------------------------------------
    # Counters
    # --------------------------------------------------------

    chunk_rows = len(valid_chunk)

    chunk_anomalies = int(
        valid_chunk[
            "is_anomaly"
        ].sum()
    )

    total_rows += chunk_rows

    total_anomalies += chunk_anomalies

    elapsed = (
        time.perf_counter()
        - chunk_start
    )

    print(
        f"Valid rows       : "
        f"{chunk_rows:,}"
    )

    print(
        f"Anomalies        : "
        f"{chunk_anomalies:,}"
    )

    print(
        f"Time             : "
        f"{elapsed:.2f}s"
    )

    print(
        f"Total processed  : "
        f"{total_rows:,}"
    )

    print(
        f"Total anomalies  : "
        f"{total_anomalies:,}"
    )

    # --------------------------------------------------------
    # Cleanup
    # --------------------------------------------------------

    del X
    del X_valid
    del X_scaled
    del X_tensor
    del chunk_scores
    del chunk_dominant_sensor
    del valid_chunk

    gc.collect()

    if device.type == "xpu":
        torch.xpu.empty_cache()

# ------------------------------------------------------------
# Final statistics
# ------------------------------------------------------------

total_time = (
    time.perf_counter()
    - start_total
)

print("\n" + "=" * 60)
print("       SKYGUARD V2 - INFERENCE COMPLETE")
print("=" * 60)

print(
    f"Total rows processed : "
    f"{total_rows:,}"
)

print(
    f"Total anomalies      : "
    f"{total_anomalies:,}"
)

print(
    f"Anomaly percentage   : "
    f"{total_anomalies / total_rows * 100:.2f}%"
)

print(
    f"Total time           : "
    f"{total_time / 60:.2f} minutes"
)

print(
    f"Results saved to     : "
    f"{OUTPUT_FILE}"
)

print("=" * 60)

print(
    "\n✅ Cell 9 completed successfully."
)

       SKYGUARD V2 - FULL DATASET INFERENCE
Dataset              : C:\Users\abhin\Downloads\archive\Indian_Weather_Dataset.csv
Output               : skyguard_v2_results.csv
Chunk size           : 2,000,000
Batch size           : 65,536
Device               : xpu
Anomaly threshold    : 0.590856

Processing chunk 1
Valid rows       : 2,000,000
Anomalies        : 20,134
Time             : 41.24s
Total processed  : 2,000,000
Total anomalies  : 20,134

Processing chunk 2
Valid rows       : 2,000,000
Anomalies        : 53,803
Time             : 29.58s
Total processed  : 4,000,000
Total anomalies  : 73,937

Processing chunk 3
Valid rows       : 2,000,000
Anomalies        : 61,941
Time             : 27.80s
Total processed  : 6,000,000
Total anomalies  : 135,878

Processing chunk 4
Valid rows       : 2,000,000
Anomalies        : 20,555
Time             : 29.37s
Total processed  : 8,000,000
Total anomalies  : 156,433

Processing chunk 5
Valid rows       : 2,000,000
Anomalies        : 81,289
Tim

In [ ]:
# ============================================================
# SKYGUARD V2 — CELL 10
# ANOMALY ANALYSIS & INTELLIGENCE
# ============================================================

import pandas as pd
import numpy as np
import os

print("=" * 60)
print("       SKYGUARD V2 - ANOMALY ANALYSIS")
print("=" * 60)

RESULT_FILE = "skyguard_v2_results.csv"

print(f"Loading: {RESULT_FILE}")

# ------------------------------------------------------------
# Load only useful columns
# ------------------------------------------------------------

analysis_columns = [
    "datetime",
    "state",
    "city",
    "lat",
    "lon",
    "anomaly_score",
    "is_anomaly",
    "severity",
    "dominant_sensor"
]

df = pd.read_csv(
    RESULT_FILE,
    usecols=analysis_columns
)

print(
    f"Rows loaded : {len(df):,}"
)

# ------------------------------------------------------------
# Convert datetime
# ------------------------------------------------------------

df["datetime"] = pd.to_datetime(
    df["datetime"],
    errors="coerce"
)

df["month"] = df["datetime"].dt.month

df["hour"] = df["datetime"].dt.hour

# ------------------------------------------------------------
# Overall statistics
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("OVERALL ANOMALY SUMMARY")
print("=" * 60)

total = len(df)

anomalies = int(
    df["is_anomaly"].sum()
)

print(
    f"Total records       : {total:,}"
)

print(
    f"Anomalies           : {anomalies:,}"
)

print(
    f"Anomaly rate        : "
    f"{anomalies / total * 100:.2f}%"
)

# ------------------------------------------------------------
# Severity distribution
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("SEVERITY DISTRIBUTION")
print("-" * 60)

severity_counts = (
    df["severity"]
    .value_counts()
)

for severity, count in severity_counts.items():

    percentage = (
        count / total * 100
    )

    print(
        f"{severity:10} : "
        f"{count:10,} "
        f"({percentage:.2f}%)"
    )

# ------------------------------------------------------------
# Dominant sensor analysis
# ------------------------------------------------------------

anomaly_df = df[
    df["is_anomaly"] == 1
].copy()

print("\n" + "-" * 60)
print("DOMINANT SENSOR IN ANOMALIES")
print("-" * 60)

sensor_counts = (
    anomaly_df[
        "dominant_sensor"
    ]
    .value_counts()
)

sensor_percentages = (
    sensor_counts
    / len(anomaly_df)
    * 100
)

sensor_summary = pd.DataFrame({
    "anomaly_count":
        sensor_counts,
    "percentage":
        sensor_percentages
})

sensor_summary = (
    sensor_summary
    .reset_index()
)

sensor_summary.columns = [
    "sensor",
    "anomaly_count",
    "percentage"
]

display(
    sensor_summary.round(2)
)

# ------------------------------------------------------------
# State analysis
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("ANOMALIES BY STATE")
print("-" * 60)

state_summary = (
    df.groupby("state")
    .agg(
        total_records=(
            "is_anomaly",
            "size"
        ),
        anomalies=(
            "is_anomaly",
            "sum"
        ),
        avg_score=(
            "anomaly_score",
            "mean"
        )
    )
)

state_summary[
    "anomaly_rate_percent"
] = (
    state_summary["anomalies"]
    / state_summary["total_records"]
    * 100
)

state_summary = (
    state_summary
    .sort_values(
        "anomaly_rate_percent",
        ascending=False
    )
)

display(
    state_summary.head(20).round(3)
)

# ------------------------------------------------------------
# City analysis
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("TOP CITIES BY ANOMALY RATE")
print("-" * 60)

city_summary = (
    df.groupby(
        ["state", "city"]
    )
    .agg(
        total_records=(
            "is_anomaly",
            "size"
        ),
        anomalies=(
            "is_anomaly",
            "sum"
        ),
        avg_score=(
            "anomaly_score",
            "mean"
        )
    )
)

city_summary[
    "anomaly_rate_percent"
] = (
    city_summary["anomalies"]
    / city_summary["total_records"]
    * 100
)

# Avoid tiny samples dominating rankings
city_summary = city_summary[
    city_summary["total_records"] >= 1000
]

city_summary = (
    city_summary
    .sort_values(
        "anomaly_rate_percent",
        ascending=False
    )
)

display(
    city_summary.head(20).round(3)
)

# ------------------------------------------------------------
# Monthly analysis
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("ANOMALIES BY MONTH")
print("-" * 60)

monthly_summary = (
    df.groupby("month")
    .agg(
        total_records=(
            "is_anomaly",
            "size"
        ),
        anomalies=(
            "is_anomaly",
            "sum"
        ),
        avg_score=(
            "anomaly_score",
            "mean"
        )
    )
)

monthly_summary[
    "anomaly_rate_percent"
] = (
    monthly_summary["anomalies"]
    / monthly_summary["total_records"]
    * 100
)

display(
    monthly_summary.round(3)
)

# ------------------------------------------------------------
# Hourly analysis
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("ANOMALIES BY HOUR")
print("-" * 60)

hourly_summary = (
    df.groupby("hour")
    .agg(
        total_records=(
            "is_anomaly",
            "size"
        ),
        anomalies=(
            "is_anomaly",
            "sum"
        ),
        avg_score=(
            "anomaly_score",
            "mean"
        )
    )
)

hourly_summary[
    "anomaly_rate_percent"
] = (
    hourly_summary["anomalies"]
    / hourly_summary["total_records"]
    * 100
)

display(
    hourly_summary.round(3)
)

# ------------------------------------------------------------
# Highest anomaly scores
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("TOP 20 MOST SEVERE ANOMALIES")
print("-" * 60)

top_anomalies = (
    anomaly_df
    .sort_values(
        "anomaly_score",
        ascending=False
    )
    .head(20)
)

display(
    top_anomalies
)

# ------------------------------------------------------------
# Save summaries
# ------------------------------------------------------------

sensor_summary.to_csv(
    "skyguard_sensor_anomaly_summary.csv",
    index=False
)

state_summary.to_csv(
    "skyguard_state_anomaly_summary.csv"
)

city_summary.to_csv(
    "skyguard_city_anomaly_summary.csv"
)

monthly_summary.to_csv(
    "skyguard_monthly_anomaly_summary.csv"
)

hourly_summary.to_csv(
    "skyguard_hourly_anomaly_summary.csv"
)

top_anomalies.to_csv(
    "skyguard_top_anomalies.csv",
    index=False
)

# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("       SKYGUARD V2 - ANALYSIS COMPLETE")
print("=" * 60)

print("Saved:")
print(" • skyguard_sensor_anomaly_summary.csv")
print(" • skyguard_state_anomaly_summary.csv")
print(" • skyguard_city_anomaly_summary.csv")
print(" • skyguard_monthly_anomaly_summary.csv")
print(" • skyguard_hourly_anomaly_summary.csv")
print(" • skyguard_top_anomalies.csv")

print("\n✅ Cell 10 completed successfully.")

In [11]:
# ============================================================
# SKYGUARD V2 - CELL 11
# TEMPORAL & PERSISTENCE ANALYSIS
# ============================================================

import pandas as pd
import numpy as np

print("=" * 60)
print("       SKYGUARD V2 - TEMPORAL ANALYSIS")
print("=" * 60)

RESULT_FILE = "skyguard_v2_results.csv"

# ------------------------------------------------------------
# Load required columns
# ------------------------------------------------------------

columns = [
    "datetime",
    "state",
    "city",
    "lat",
    "lon",
    "anomaly_score",
    "is_anomaly",
    "severity",
    "dominant_sensor"
]

print("Loading anomaly results...")

df = pd.read_csv(
    RESULT_FILE,
    usecols=columns
)

print(
    f"Rows loaded : {len(df):,}"
)

# ------------------------------------------------------------
# Datetime
# ------------------------------------------------------------

df["datetime"] = pd.to_datetime(
    df["datetime"],
    errors="coerce"
)

# ------------------------------------------------------------
# Sort geographically + chronologically
# ------------------------------------------------------------

df = df.sort_values(
    ["state", "city", "datetime"]
).reset_index(drop=True)

print("Data sorted by location and time.")

# ------------------------------------------------------------
# Detect consecutive anomalous observations
# ------------------------------------------------------------

df["anomaly_binary"] = (
    df["is_anomaly"] == 1
).astype(int)

# A new group starts whenever anomaly status changes
df["group_change"] = (
    df["anomaly_binary"]
    != df["anomaly_binary"].shift()
).astype(int)

df["group_id"] = (
    df["group_change"].cumsum()
)

# ------------------------------------------------------------
# Calculate run length
# ------------------------------------------------------------

run_lengths = (
    df.groupby("group_id")[
        "anomaly_binary"
    ]
    .transform("sum")
)

df["anomaly_run_length"] = (
    run_lengths
)

# Only meaningful for anomalies
df.loc[
    df["is_anomaly"] == 0,
    "anomaly_run_length"
] = 0

# ------------------------------------------------------------
# Persistent anomalies
# ------------------------------------------------------------

persistent = df[
    (df["is_anomaly"] == 1) &
    (df["anomaly_run_length"] >= 3)
].copy()

print("\n" + "=" * 60)
print("PERSISTENT ANOMALIES")
print("=" * 60)

print(
    f"Total anomalous records      : "
    f"{df['is_anomaly'].sum():,}"
)

print(
    f"Persistent anomaly records   : "
    f"{len(persistent):,}"
)

if df["is_anomaly"].sum() > 0:

    print(
        f"Persistent percentage       : "
        f"{len(persistent) / df['is_anomaly'].sum() * 100:.2f}%"
    )

# ------------------------------------------------------------
# Run distribution
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("ANOMALY RUN DISTRIBUTION")
print("-" * 60)

anomaly_runs = (
    df[
        df["is_anomaly"] == 1
    ]
    .groupby("group_id")
    .size()
)

print(
    f"Total anomaly runs : "
    f"{len(anomaly_runs):,}"
)

print(
    f"Average run length : "
    f"{anomaly_runs.mean():.2f}"
)

print(
    f"Maximum run length : "
    f"{anomaly_runs.max():,}"
)

print(
    f"Runs >= 2          : "
    f"{(anomaly_runs >= 2).sum():,}"
)

print(
    f"Runs >= 3          : "
    f"{(anomaly_runs >= 3).sum():,}"
)

print(
    f"Runs >= 6          : "
    f"{(anomaly_runs >= 6).sum():,}"
)

print(
    f"Runs >= 12         : "
    f"{(anomaly_runs >= 12).sum():,}"
)

# ------------------------------------------------------------
# Top persistent events
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TOP PERSISTENT ANOMALY EVENTS")
print("=" * 60)

persistent_events = (
    df[
        df["is_anomaly"] == 1
    ]
    .groupby("group_id")
    .agg(
        start_time=(
            "datetime",
            "min"
        ),
        end_time=(
            "datetime",
            "max"
        ),
        duration_records=(
            "datetime",
            "size"
        ),
        state=(
            "state",
            "first"
        ),
        city=(
            "city",
            "first"
        ),
        max_score=(
            "anomaly_score",
            "max"
        ),
        avg_score=(
            "anomaly_score",
            "mean"
        ),
        dominant_sensor=(
            "dominant_sensor",
            lambda x:
            x.value_counts().index[0]
        ),
        severity=(
            "severity",
            lambda x:
            x.value_counts().index[0]
        )
    )
)

persistent_events = persistent_events[
    persistent_events["duration_records"] >= 3
]

persistent_events = (
    persistent_events
    .sort_values(
        "max_score",
        ascending=False
    )
)

display(
    persistent_events.head(30)
)

# ------------------------------------------------------------
# Persistent anomalies by sensor
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("PERSISTENT ANOMALIES BY SENSOR")
print("-" * 60)

persistent_sensor = (
    persistent[
        "dominant_sensor"
    ]
    .value_counts()
)

persistent_sensor_percent = (
    persistent_sensor
    / len(persistent)
    * 100
)

persistent_sensor_summary = pd.DataFrame({
    "persistent_anomalies":
        persistent_sensor,
    "percentage":
        persistent_sensor_percent
})

persistent_sensor_summary = (
    persistent_sensor_summary
    .reset_index()
)

persistent_sensor_summary.columns = [
    "sensor",
    "persistent_anomalies",
    "percentage"
]

display(
    persistent_sensor_summary.round(2)
)

# ------------------------------------------------------------
# Persistent anomalies by state
# ------------------------------------------------------------

print("\n" + "-" * 60)
print("PERSISTENT ANOMALIES BY STATE")
print("-" * 60)

persistent_state = (
    persistent
    .groupby("state")
    .size()
    .sort_values(
        ascending=False
    )
)

display(
    persistent_state.head(20)
)

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

persistent_events.to_csv(
    "skyguard_persistent_events.csv"
)

persistent_sensor_summary.to_csv(
    "skyguard_persistent_sensor_summary.csv",
    index=False
)

persistent.to_csv(
    "skyguard_persistent_records.csv",
    index=False
)

# ------------------------------------------------------------
# Final
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SKYGUARD V2 - TEMPORAL ANALYSIS COMPLETE")
print("=" * 60)

print("Saved:")
print(" • skyguard_persistent_events.csv")
print(" • skyguard_persistent_sensor_summary.csv")
print(" • skyguard_persistent_records.csv")

print("\n✅ Cell 11 completed successfully.")

       SKYGUARD V2 - TEMPORAL ANALYSIS
Loading anomaly results...
Rows loaded : 46,082,160
Data sorted by location and time.

PERSISTENT ANOMALIES
Total anomalous records      : 1,056,291
Persistent anomaly records   : 628,987
Persistent percentage       : 59.55%

------------------------------------------------------------
ANOMALY RUN DISTRIBUTION
------------------------------------------------------------
Total anomaly runs : 437,975
Average run length : 2.41
Maximum run length : 549
Runs >= 2          : 168,755
Runs >= 3          : 89,713
Runs >= 6          : 34,250
Runs >= 12         : 9,267

TOP PERSISTENT ANOMALY EVENTS


,start_time,end_time,duration_records,state,city,max_score,avg_score,dominant_sensor,severity
group_id,,,,,,,,,
130886,2019-08-09 18:00:00,2019-08-10 05:00:00,12,Gujarat,Rajkot,162.209870,50.661345,wind_dir_cos,Critical
125052,2017-07-22 05:00:00,2017-07-22 12:00:00,8,Gujarat,Ahmedabad,61.144573,20.707274,wind_dir_cos,Critical
370318,2022-05-19 03:00:00,2022-05-19 05:00:00,3,Karnataka,Mangaluru,60.820670,41.101861,wind_dir_cos,Critical
669620,2021-08-03 05:00:00,2021-08-03 11:00:00,7,Rajasthan,Kota,55.208430,14.776241,wind_dir_cos,Critical
752452,2023-12-17 19:00:00,2023-12-18 08:00:00,14,Tamil_Nadu,Tirunelveli,52.696940,12.674708,wind_dir_cos,Critical
522286,2025-05-30 00:00:00,2025-05-30 19:00:00,20,Meghalaya,Cherrapunji,52.299892,8.510109,wind_dir_cos,Critical
110270,2018-06-25 06:00:00,2018-06-25 13:00:00,8,Dadra_Nagar_Haveli,Silvassa,48.250713,16.231057,wind_dir_cos,Critical
120060,2018-06-08 12:00:00,2018-06-08 17:00:00,6,Goa,Panaji,43.811270,10.503838,wind_dir_cos,Critical
808466,2024-07-03 07:00:00,2024-07-03 12:00:00,6,Uttarakhand,Almora,43.176170,16.149362,wind_dir_cos,Critical



------------------------------------------------------------
PERSISTENT ANOMALIES BY SENSOR
------------------------------------------------------------


,sensor,persistent_anomalies,percentage
0,wind_speed_ms,203092,32.29
1,pressure_trend,109995,17.49
2,cloud_cover_pct,95158,15.13
3,temperature_C,93770,14.91
4,wind_dir_cos,51213,8.14
5,pressure_hPa,43164,6.86
6,dew_point_C,23548,3.74
7,wind_dir_sin,4889,0.78
8,solar_radiation_Wm2,2087,0.33
9,precip_mm,1855,0.29



------------------------------------------------------------
PERSISTENT ANOMALIES BY STATE
------------------------------------------------------------


state
Ladakh              211371
Jammu_Kashmir        65112
Tamil_Nadu           47613
Andaman_Nicobar      41542
Himachal_Pradesh     37295
Karnataka            26485
Punjab               22077
Meghalaya            18364
Rajasthan            18284
Haryana              16204
Lakshadweep          12651
Uttar_Pradesh        12631
Kerala                9209
Uttarakhand           7984
Sikkim                7791
West_Bengal           7003
Bihar                 6357
Puducherry            5967
Odisha                5823
Jharkhand             5701
dtype: int64


SKYGUARD V2 - TEMPORAL ANALYSIS COMPLETE
Saved:
 • skyguard_persistent_events.csv
 • skyguard_persistent_sensor_summary.csv
 • skyguard_persistent_records.csv

✅ Cell 11 completed successfully.
